<a href="https://colab.research.google.com/github/kinshukpcs24-ux/AI_Lab_Programs/blob/main/1BM24CS137_Week7_First_Order_Logic/1BM24CS137_Week7_First_Order_Logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# Helper to check if an expression is a variable
def is_variable(expr):
    # Any string that is not a compound function structure is treated as a variable/constant to allow unification
    return isinstance(expr, str) and not is_function(expr)

# Helper to check if an expression is a compound term/function
def is_function(expr):
    return isinstance(expr, dict) and 'name' in expr and 'args' in expr

# Recursively applies the current substitutions to a term
def apply_substitution(term, subst):
    if is_variable(term):
        if term in subst:
            return apply_substitution(subst[term], subst)
        return term
    if is_function(term):
        return {
            'name': term['name'],
            'args': [apply_substitution(arg, subst) for arg in term['args']]
        }
    return term

# Converts internal dictionary format back to readable string
def to_string(term):
    if is_variable(term):
        return str(term)
    if is_function(term):
        args_str = ", ".join(to_string(arg) for arg in term['args'])
        return f"{term['name']}({args_str})"
    return str(term)

# Occurs check: ensures we do not unify a variable with a term containing that variable
def occurs_check(var, term, subst):
    if var == term:
        return True
    elif is_variable(term):
        if term in subst:
            return occurs_check(var, subst[term], subst)
    elif is_function(term):
        return any(occurs_check(var, arg, subst) for arg in term['args'])
    return False

# Unify a single variable with any other term
def unify_variable(var, term, subst, reasons):
    if var == term:
        return subst
    elif var in subst:
        return unify(subst[var], term, subst, reasons)
    elif is_variable(term) and term in subst:
        return unify(var, subst[term], subst, reasons)
    elif occurs_check(var, term, subst):
        reasons.append(f"Occurs check failed: variable '{var}' occurs inside '{to_string(term)}'")
        return None
    else:
        new_subst = subst.copy()
        new_subst[var] = term
        return new_subst

# Main unification algorithm with substitution tracking and composition
def unify(x, y, subst=None, reasons=None):
    if reasons is None:
        reasons = []
    if subst is None:
        subst = {}

    # Apply current substitution bindings to both terms before evaluating
    x = apply_substitution(x, subst)
    y = apply_substitution(y, subst)

    if x == y:
        return subst
    if is_variable(x):
        return unify_variable(x, y, subst, reasons)
    if is_variable(y):
        return unify_variable(y, x, subst, reasons)
    if is_function(x) and is_function(y):
        if x['name'] != y['name']:
            reasons.append(f"Function name mismatch: '{x['name']}' vs '{y['name']}'")
            return None
        if len(x['args']) != len(y['args']):
            reasons.append(f"Function argument count mismatch: '{x['name']}' has {len(x['args'])} args vs '{y['name']}' with {len(y['args'])} args")
            return None
        current_subst = subst
        for arg_x, arg_y in zip(x['args'], y['args']):
            current_subst = unify(arg_x, arg_y, current_subst, reasons)
            if current_subst is None:
                return None
        return current_subst

    reasons.append(f"Clash: cannot unify '{to_string(x)}' with '{to_string(y)}'")
    return None

# Structural parser designed to automatically detect function names from brackets
def parse_term(expr_str):
    expr_str = expr_str.strip()
    if '(' not in expr_str:
        return expr_str

    # Name is everything before the first opening bracket
    name, rest = expr_str.split('(', 1)
    name = name.strip()
    rest = rest.rstrip(')').strip()

    args = []
    bracket_count = 0
    current_arg = []
    for char in rest:
        if char == ',' and bracket_count == 0:
            args.append(parse_term("".join(current_arg)))
            current_arg = []
        else:
            if char == '(':
                bracket_count += 1
            elif char == ')':
                bracket_count -= 1
            current_arg.append(char)
    if current_arg:
        args.append(parse_term("".join(current_arg)))

    return {'name': name, 'args': args}

# Dynamic input block with structural parser
print("=== FOL Unification Solver ===")
try:
    str1 = input("Expression 1: ")
    str2 = input("Expression 2: ")

    t1 = parse_term(str1)
    t2 = parse_term(str2)

    reasons = []
    result = unify(t1, t2, reasons=reasons)
    print(f"\nUnifying: {str1} and {str2}")
    if result is None:
        print("MGU: None (Unification Failed)")
        print("Reason for failure:")
        for r in reasons:
            print(f" - {r}")
    else:
        # Format final bindings as clean FOL text representations
        readable_mgu = {str(k): to_string(v) for k, v in result.items()}
        print(f"MGU: {readable_mgu}")
except Exception as e:
    print(f"Error parsing expressions: {e}")

=== FOL Unification Solver ===
Expression 1: p(b, X, f(g(Z)))
Expression 2: p(Z, f(Y), f(Y))

Unifying: p(b, X, f(g(Z))) and p(Z, f(Y), f(Y))
MGU: {'b': 'Z', 'X': 'f(Y)', 'Y': 'g(Z)'}
